In [8]:
from dotenv import load_dotenv

load_dotenv()

from typing import Any, List, Optional, Dict
import os

from langchain_core.callbacks import CallbackManagerForLLMRun
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import (
    AIMessage,
    BaseMessage,
    HumanMessage,
    SystemMessage,
    ChatMessage,
)
from langchain_core.outputs import ChatGeneration, ChatResult
from openai import OpenAI
from pydantic import Field, PrivateAttr


class QwenChatModel(BaseChatModel):
    """
    基于 BaseChatModel 封装的阿里云 Qwen 自定义类。
    支持 enable_thinking 参数以获取思考过程。
    """

    model_name: str = Field(default="qwen-plus", alias="model")
    api_key: Optional[str] = Field(default=None)
    base_url: Optional[str] = Field(default=None)

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.api_key = self.api_key or os.getenv("OPENAI_API_KEY")
        self.base_url = self.base_url or os.getenv("OPENAI_API_BASE")
        if not self.api_key or not self.base_url:
            raise ValueError("API key 和 Base URL 必须通过参数或环境变量提供")

        # 初始化 OpenAI 原生客户端
        self._client = OpenAI(
            api_key=self.api_key,
            base_url=self.base_url,
        )

    @property
    def _llm_type(self) -> str:
        return "qwen_dashscope"

    def _convert_messages_to_openai_format(self, messages: List[BaseMessage]) -> List[Dict[str, Any]]:
        """将 LangChain 消息格式转换为 OpenAI SDK 要求的格式"""
        openai_messages = []
        for msg in messages:
            role = "user"
            if isinstance(msg, HumanMessage):
                role = "user"
            elif isinstance(msg, AIMessage):
                role = "assistant"
            elif isinstance(msg, SystemMessage):
                role = "system"
            elif isinstance(msg, ChatMessage):
                role = msg.role

            openai_messages.append({"role": role, "content": msg.content})
        return openai_messages

    def _generate(
            self,
            messages: List[BaseMessage],
            stop: Optional[List[str]] = None,
            run_manager: Optional[CallbackManagerForLLMRun] = None,
            **kwargs: Any,
    ) -> ChatResult:
        """核心生成逻辑"""

        # 1. 转换消息格式
        openai_messages = self._convert_messages_to_openai_format(messages)

        # 2. 准备请求参数 (extra_body 是关键)
        extra_body = kwargs.pop("extra_body", {})

        # 3. 调用 OpenAI SDK
        response = self._client.chat.completions.create(
            model=self.model_name,
            messages=openai_messages,
            extra_body=extra_body,  # 传入百炼特有参数
            stop=stop,
            **kwargs
        )

        # 4. 解析结果
        choice = response.choices[0]
        message = choice.message
        content = message.content

        # 5. 处理思考过程 (Reasoning Content)
        # 百炼 API 的思考内容通常在 message 的 reasoning_content 字段中（如果有）
        # 或者有时在 extra_fields 里，我们将其放入 additional_kwargs 以便后续查看
        reasoning_content = getattr(message, "reasoning_content", None)

        additional_kwargs = {}
        if reasoning_content:
            additional_kwargs["reasoning_content"] = reasoning_content

        # 6. 构造 LangChain 的返回值
        generations = [
            ChatGeneration(
                message=AIMessage(
                    content=content,
                    additional_kwargs=additional_kwargs  # 思考过程存在这里
                )
            )
        ]

        return ChatResult(generations=generations)

In [9]:
llm = QwenChatModel(
    model="qwen-plus"
)

In [11]:
from langchain_core.messages import HumanMessage
from IPython.display import display, Markdown

messages = [
    HumanMessage(content="请帮我解方程：x^2 - 5x + 6 = 0")
]
result = llm.invoke(messages, extra_body={'enable_thinking': True})

reasoning = result.additional_kwargs.get("reasoning_content", "")
final_answer = result.content

# 渲染思考过程
print("=== 思考过程 ===")
if reasoning:
    display(Markdown(f"> {reasoning}"))  # 加个引用符号区分
else:
    print("无思考过程")

print("\n=== 最终回答 ===")
display(Markdown(final_answer))

=== 思考过程 ===


> 我现在要解这个二次方程x² - 5x + 6 = 0。首先，我记得二次方程的一般解法有因式分解、求根公式、配方法等等。先试试看能不能因式分解，因为如果能分解的话会比较快。

首先，二次项的系数是1，所以分解的时候应该是(x + a)(x + b) = x² + (a + b)x + ab。这里的方程是x² - 5x + 6，所以需要找到两个数a和b，使得它们的和是-5，积是6。不过可能更直观的是，因为一次项系数是-5，常数项是+6，所以应该是两个负数相乘得正6，相加得-5。或者说，找两个数，它们的和是5（因为一次项是-5x，所以分解后应该是(x - m)(x - n)，其中m + n = 5，m*n = 6）。

对的，应该是(x - m)(x - n) = x² - (m + n)x + mn，所以这里mn=6，m + n=5。那找两个数，乘积6，和5。可能的组合是2和3，因为2*3=6，2+3=5。所以分解的话就是(x - 2)(x - 3)=0。

接下来，根据零乘积定理，如果两个数的乘积为0，那么至少其中一个数为0。所以x - 2=0或者x - 3=0，解得x=2或x=3。

为了验证一下是否正确，可以代入原方程看看。当x=2时，左边是2² - 5*2 + 6 = 4 - 10 + 6 = 0，正确。当x=3时，左边是3² - 5*3 + 6 = 9 - 15 + 6 = 0，也正确。

另外，也可以用求根公式来验证。二次方程ax² + bx + c = 0的求根公式是x = [-b ± √(b² - 4ac)]/(2a)。这里a=1，b=-5，c=6，所以判别式D = (-5)² - 4*1*6 = 25 - 24 = 1，根号D=1。所以x = [5 ± 1]/2，即(5+1)/2=3，(5-1)/2=2，和因式分解的结果一致。

或者用配方法，x² - 5x + 6 = 0，移项得x² - 5x = -6。配方的话，x² - 5x + (25/4) = -6 + 25/4，左边是(x - 5/2)²，右边是(-24/4 + 25/4)=1/4。所以x - 5/2 = ±√(1/4)=±1/2，解得x=5/2 + 1/2=3，x=5/2 - 1/2=2，同样得到两个解。

所以无论是因式分解、求根公式还是配方法，结果都是x=2和x=3，说明这两个解是正确的。


=== 最终回答 ===


要解方程 $x^2 - 5x + 6 = 0$，可以通过以下步骤进行：

---

### **1. 因式分解法**
观察方程 $x^2 - 5x + 6 = 0$，尝试将其分解为两个一次因式的乘积：
$$
x^2 - 5x + 6 = (x - 2)(x - 3)
$$
验证：
$$
(x - 2)(x - 3) = x^2 - 3x - 2x + 6 = x^2 - 5x + 6
$$
成立。

根据**零乘积定理**，若 $(x - 2)(x - 3) = 0$，则：
$$
x - 2 = 0 \quad \text{或} \quad x - 3 = 0
$$
解得：
$$
x = 2 \quad \text{或} \quad x = 3
$$

---

### **2. 求根公式法**
使用二次方程求根公式：
$$
x = \frac{-b \pm \sqrt{b^2 - 4ac}}{2a}
$$
其中 $a = 1$，$b = -5$，$c = 6$。代入得：
$$
x = \frac{-(-5) \pm \sqrt{(-5)^2 - 4 \cdot 1 \cdot 6}}{2 \cdot 1} = \frac{5 \pm \sqrt{25 - 24}}{2} = \frac{5 \pm 1}{2}
$$
计算得：
$$
x = \frac{5 + 1}{2} = 3 \quad \text{或} \quad x = \frac{5 - 1}{2} = 2
$$

---

### **3. 配方法**
将方程配方：
$$
x^2 - 5x + 6 = 0 \implies x^2 - 5x = -6
$$
配方：$x^2 - 5x + \left(\frac{5}{2}\right)^2 = -6 + \left(\frac{5}{2}\right)^2$
$$
\left(x - \frac{5}{2}\right)^2 = \frac{1}{4}
$$
开平方：
$$
x - \frac{5}{2} = \pm \frac{1}{2} \implies x = \frac{5}{2} \pm \frac{1}{2}
$$
解得：
$$
x = 3 \quad \text{或} \quad x = 2
$$

---

### **最终答案**
方程的解为：
$$
\boxed{x = 2 \quad \text{或} \quad x = 3}
$$